# 🧠 Fine-Tuning BanglaBERT on IndoWordNet for Bengali WSD (Gloss Cross-Encoder, v1)

### What was wrong in v0 (68% test accuracy) and what this version fixes

| Problem in v0 | Why it hurt | Fix in v1 |
|---|---|---|
| **Glosses were truncated** by the dataset generator (≤ 6 words, cut at `যা/যার/যে…`), e.g. `দড়ি` sense 6 = `"ধরনের মোটা দড়ি"` | **Every val/test sense has 0 training examples** (IndoWordNet has ~1 example per synset), so the model can *only* succeed by reading the gloss. Half-glosses → guesswork. | **Step 1b** re-fetches the **full IndoWordNet gloss + synonyms** for every sense, with a strict alignment check so labels stay correct. |
| Only ~6k training sentences | Too few (context, gloss) pairs to learn gloss matching | **Step 1b** adds ~11k extra IndoWordNet sentences from words *not* in the catalog, with a leakage guard (no shared synsets / sentences with val/test). |
| No BanglaBERT text normalisation | BanglaBERT was pre-trained on text normalised by `csebuetnlp/normalizer`; the authors warn un-normalised input degrades results | Normaliser applied to both sentence and gloss. |
| Target marked as `**word**` (4 junk `*` tokens) | Weak target signal | GlossBERT-style quote marking: `... " word " ...` |
| `truncation="only_first"`, 256 tokens | Could cut the target out of the sentence | `longest_first`, 128 tokens (all pairs fit). |
| Stray forward pass before the loop, deprecated `torch.cuda.amp` | Wasted compute / warnings | Removed / updated. |
| 3 epochs, no early stopping | Under-trained (train acc 79%) | Up to 8 epochs, early stopping on val accuracy. |
| **Inference used an unmarked sentence** and looked up `ফল` which is not in the catalog → `IndexError` | Train/inference mismatch | `mark_target()` finds the (inflected) target in raw sentences; senses come from catalog or IndoWordNet. |

### Cross-encoder input
`[CLS]  sentence with " target "  [SEP]  target : full gloss ; সমার্থক: syn1, syn2  [SEP]` → linear head → 1 score per candidate → softmax over the word's senses → cross-entropy.

## Step 0: Environment Setup
Installs `transformers`, `pyiwn` (IndoWordNet) and the official BanglaBERT `normalizer`.

In [ ]:
import os, sys, re, json, random, time, unicodedata, collections, subprocess
from pathlib import Path
os.environ["PYTHONUTF8"] = "1"

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import transformers, torch
except ImportError:
    _pip("transformers", "torch", "accelerate"); import transformers, torch
try:
    import pyiwn
except ImportError:
    _pip("pyiwn"); import pyiwn
try:
    from normalizer import normalize as bn_normalize
except ImportError:
    try:
        _pip("git+https://github.com/csebuetnlp/normalizer")
        from normalizer import normalize as bn_normalize
    except Exception as e:
        print("WARNING: BanglaBERT normalizer unavailable, falling back to NFC only:", e)
        bn_normalize = lambda s: unicodedata.normalize("NFC", s)

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"PyTorch {torch.__version__} | Transformers {transformers.__version__} | device={device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1: Load the processed IndoWordNet dataset
Also prints the two baselines the model has to beat, and checks how many val/test senses were ever seen in training.

In [ ]:
dataset_path = Path("./dataset_splits.json")
if not dataset_path.exists():
    dataset_path = Path("../data/processed_indowordnet/dataset_splits.json")
with open(dataset_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

catalog = dataset["catalog"]
train_records, val_records, test_records = dataset["train"], dataset["val"], dataset["test"]

# Labels must index into the *ordered* list of sense keys - verify instead of assuming
for r in train_records + val_records + test_records:
    assert list(catalog[r["folder"]]["senses"]).index(str(r["sense_num"])) == r["sense_label"]

seen_senses = {(r["folder"], r["sense_num"]) for r in train_records}
def _baselines(recs):
    rand = sum(1 / len(catalog[r["folder"]]["senses"]) for r in recs) / len(recs)
    first = sum(r["sense_label"] == 0 for r in recs) / len(recs)
    seen = sum((r["folder"], r["sense_num"]) in seen_senses for r in recs) / len(recs)
    return rand, first, seen

print(f"Catalog words: {len(catalog):,} | train {len(train_records):,} | val {len(val_records):,} | test {len(test_records):,}")
for name, recs in [("val", val_records), ("test", test_records)]:
    rand, first, seen = _baselines(recs)
    print(f"{name:>4}: random = {rand*100:.1f}% | first-sense = {first*100:.1f}% | sense seen in train = {seen*100:.1f}%")
print("\n-> val/test senses are (almost) never seen in training: the model MUST learn to read glosses.")

## Step 1b: Repair glosses & add extra IndoWordNet training data
1. **Full glosses** – For each catalog word we re-read its synsets from IndoWordNet, re-apply the generator's exact filtering, and **only if the old (truncated) glosses are reproduced exactly in the same order** we replace them with the full definition + up to 3 synonyms. This guarantees `sense_num`/`sense_label` stay valid.
2. **Extra training data** – polysemous IndoWordNet words that are *not* in the catalog. A word is skipped if any of its synsets is a val/test gold sense, and any sentence identical to a val/test sentence is dropped (no leakage).

Set `USE_FULL_GLOSSES` / `USE_EXTRA_TRAIN` to `False` to ablate.

In [ ]:
# ---- Same normalisation / filtering the dataset generator used (needed to re-align sense numbers) ----
_BENGALI_CHAR = r"ঀ-৿"
_OBSCURE_TERMS = ['অসুর', 'বিরাটের পুত্র', 'বৈদীক যুগের', 'একটি কাব্যালঙ্কার']

def normalize_text_clean(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[﻿​‌‍\r\t]", " ", text)
    text = text.replace("_", " ")
    text = re.sub(r'["`~^+=|\\/«»]', " ", text)
    text = text.replace("নদী বী ", "নদী বা ")
    text = text.replace("অবস্হিত", "অবস্থিত").replace("অবস্হা", "অবস্থা")
    text = text.replace("মুখথেকে", "মুখ থেকে").replace("ব্যাক্তি", "ব্যক্তি")
    text = re.sub(r"\s+", " ", text).strip(" _-—\t\r\n")
    return text

def clean_gloss_normalized(raw_gloss: str, lemmas: list, target_word: str) -> str:
    """Exact copy of the generator's (lossy) gloss cleaner - used ONLY to verify sense alignment."""
    g = normalize_text_clean(raw_gloss)
    g = re.sub(r'^\s*\([^)]+\)\s*', '', g)
    if 'ফলস্বরূপ হওয়া' in g or 'শেষে তার' in g:
        g = 'কাজের শেষ পরিণতি বা ফলাফল'
    elif 'ফুল থেকে উত্পন্ন হওয়া শাঁস' in g:
        g = 'গাছের রসালো খাদ্য বা বীজকোষ'
    elif 'পরিণাম রূপে প্রাপ্ত ফল' in g or g == 'পরিণাম রূপে প্রাপ্ত':
        g = 'কর্মের প্রতিফল বা বদলা'
    elif 'গণিতে কোনো সমস্যার' in g:
        g = 'গণিতের সমাধান বা প্রশ্নের উত্তর'
    else:
        prefixes = [r'^কোনো এমন বস্তু যা\s*', r'^এমন বস্তু যা\s*', r'^এমন বিষয় যা\s*', r'^সেই প্রধান\s*',
                    r'^মানুষের সেই সমূহ যাদের কাছে\s*', r'^সেই\s+', r'^কোনো\s+', r'^কোনও\s+',
                    r'^একপ্রকার\s+', r'^একটি\s+', r'^একজন\s+', r'^এক\s+']
        for p in prefixes:
            g = re.sub(p, '', g, flags=re.IGNORECASE).strip()
        parts = re.split(r'\s+(?:যা|যার|যাকে|যাদের|যাতে|যেখানে|যখন|যে সময়|এবং যার)\s+', g)
        if parts[0] and len(parts[0].split()) >= 2:
            g = parts[0].strip()
        elif len(parts) > 1 and parts[1]:
            g = parts[1].strip()
    g = re.sub(r'\s+(?:বা|এবং|অথবা|ও|ইত্যাদি|প্রভৃতি|সেই)$', '', g).strip(' ,;:-—')
    words = g.split()
    if len(words) > 6:
        g = ' '.join(words[:6])
    norm_target = normalize_text_clean(target_word)
    other = [l for l in (normalize_text_clean(l) for l in lemmas) if l.lower() != norm_target.lower()]
    seen = set()
    uniq = [l for l in other if not (l in seen or seen.add(l))]
    if uniq:
        syn_str = ', '.join(uniq[:2])
        if g:
            return g if g.startswith(syn_str) else f"{syn_str} ({g})"
        return syn_str
    return g

def mark_target_in_sentence(context: str, target: str, lemmas: list):
    norm_context = normalize_text_clean(context)
    for cand in [target] + list(lemmas):
        c = normalize_text_clean(cand)
        if not c:
            continue
        pat = rf'(?<![{_BENGALI_CHAR}]){re.escape(c)}([{_BENGALI_CHAR}]*)(?![{_BENGALI_CHAR}])'
        if re.search(pat, norm_context):
            return re.sub(pat, rf'**{c}\1**', norm_context, count=1), True
    return norm_context, False

# ---- NEW: full, untruncated gloss = synonyms + complete IndoWordNet definition ----
def full_gloss(synset, target_word: str, max_syns: int = 3) -> str:
    gloss = normalize_text_clean(synset.gloss())
    t = normalize_text_clean(target_word)
    syns, seen = [], set()
    for l in synset.lemma_names():
        l = normalize_text_clean(l)
        if l and l != t and l not in seen:
            seen.add(l); syns.append(l)
    syns = syns[:max_syns]
    if syns and gloss:
        return f"{gloss} ; সমার্থক: {', '.join(syns)}"
    return gloss or ', '.join(syns) or t

def filtered_synsets(iwn, word: str):
    for w in (word, word.replace(" ", "_")):
        try:
            syns = iwn.synsets(w)
        except Exception:
            continue
        syns = [s for s in syns if not any(t in s.gloss() for t in _OBSCURE_TERMS)]
        if syns:
            return syns
    return []

def enrich_catalog(iwn, catalog):
    """Replace truncated glosses with full ones, but ONLY when the synset order provably matches
    the stored sense numbers (so dataset labels stay valid)."""
    new_catalog, stats = {}, collections.Counter()
    for wid, entry in catalog.items():
        tw, old = entry["target_word"], entry["senses"]
        syns = filtered_synsets(iwn, tw)
        ok = len(syns) == len(old) and all(
            clean_gloss_normalized(s.gloss(), s.lemma_names(), tw) == old[str(i)]
            for i, s in enumerate(syns, 1))
        if ok:
            new_catalog[wid] = {"target_word": tw,
                                "senses": {str(i): full_gloss(s, tw) for i, s in enumerate(syns, 1)},
                                "synset_ids": {str(i): s.synset_id() for i, s in enumerate(syns, 1)}}
            stats["aligned"] += 1
        else:
            new_catalog[wid] = entry
            stats["kept_old"] += 1
    return new_catalog, dict(stats)

def build_extra_train(iwn, dataset, max_sentences=30000, excluded_words=()):
    """Extra training sentences from IndoWordNet words that are NOT in the catalog.
    Leakage guard: skip any word that shares a synset or an example sentence with val/test."""
    catalog = dataset["catalog"]
    heldout = dataset["val"] + dataset["test"]
    strip = lambda s: normalize_text_clean(s.replace("**", ""))
    heldout_texts = {strip(r["text"]) for r in heldout}
    heldout_syn_ids = set()
    for r in heldout:
        ids = catalog[r["folder"]].get("synset_ids")
        if ids:
            heldout_syn_ids.add(ids[str(r["sense_num"])])
    used_words = {v["target_word"] for v in catalog.values()} | set(excluded_words)

    extra_catalog, extra_records = {}, []
    for w in iwn.all_words():
        if len(extra_records) >= max_sentences:
            break
        tw = normalize_text_clean(w)
        if not tw or tw in used_words:
            continue
        used_words.add(tw)
        syns = filtered_synsets(iwn, w)
        if len(syns) < 2 or any(s.synset_id() in heldout_syn_ids for s in syns):
            continue
        wid = f"IWN_Extra_{len(extra_catalog) + 1}"
        senses = {str(i): full_gloss(s, tw) for i, s in enumerate(syns, 1)}
        recs = []
        for i, s in enumerate(syns, 1):
            for ex in s.examples():
                if strip(ex) in heldout_texts:
                    continue
                marked, found = mark_target_in_sentence(ex, tw, s.lemma_names())
                if found:
                    recs.append({"folder": wid, "target_word": tw, "sense_num": i, "sense_label": i - 1,
                                 "text": marked, "num_senses_for_word": len(syns), "split": "train_extra"})
        if recs:
            extra_catalog[wid] = {"target_word": tw, "senses": senses}
            extra_records.extend(recs)
    return extra_catalog, extra_records

In [ ]:
USE_FULL_GLOSSES = True
USE_EXTRA_TRAIN = True
MAX_EXTRA_SENTENCES = 30000

iwn = pyiwn.IndoWordNet(pyiwn.Language.BENGALI)

old_catalog = catalog
if USE_FULL_GLOSSES:
    catalog, stats = enrich_catalog(iwn, old_catalog)
    print(f"Gloss repair: {stats}")
    dataset["catalog"] = catalog

extra_catalog, extra_train = {}, []
if USE_EXTRA_TRAIN:
    extra_catalog, extra_train = build_extra_train(iwn, dataset, MAX_EXTRA_SENTENCES)
    print(f"Extra IndoWordNet training data: {len(extra_catalog):,} words, {len(extra_train):,} sentences")

ALL_SENSES = {k: v["senses"] for k, v in {**catalog, **extra_catalog}.items()}
full_train = train_records + extra_train
print(f"Total training sentences: {len(full_train):,}")

wid = test_records[0]["folder"]
print(f"\nExample - '{catalog[wid]['target_word']}':")
for n in catalog[wid]["senses"]:
    print(f"  [old] {old_catalog[wid]['senses'][n]}")
    print(f"  [new] {catalog[wid]['senses'][n]}")

# Save the repaired dataset so it can be reused without pyiwn
out = dataset_path.parent / "dataset_splits_fullgloss.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump({**dataset, "extra_catalog": extra_catalog, "train_extra": extra_train}, f, ensure_ascii=False)
print(f"\nSaved repaired dataset -> {out}")

## Step 2: Cross-encoder pair construction
* Sentence: BanglaBERT-normalised, target wrapped in quotes (`**দড়ি**` → `" দড়ি "`).
* Gloss side: `target : full gloss ; সমার্থক: …`

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "csebuetnlp/banglabert"
MAX_LEN = 128
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

_MARK = re.compile(r"\*\*(.+?)\*\*")

def prepare_context(marked_text: str) -> str:
    # '**word**' -> ' " word " ' (GlossBERT-style weak supervision marker)
    text = _MARK.sub(lambda m: f' " {m.group(1).strip()} " ', marked_text)
    return re.sub(r"\s+", " ", bn_normalize(text)).strip()

def prepare_gloss(target_word: str, definition: str) -> str:
    return bn_normalize(f"{target_word} : {definition}")

def build_cross_encoder_pairs(context: str, target_word: str, senses: dict):
    ctx = prepare_context(context)
    firsts, seconds, nums = [], [], []
    for num, definition in senses.items():
        firsts.append(ctx)
        seconds.append(prepare_gloss(target_word, definition))
        nums.append(int(num))
    return firsts, seconds, nums

def encode_pairs(firsts, seconds):
    return tokenizer(firsts, seconds, max_length=MAX_LEN, truncation="longest_first",
                     padding=True, return_tensors="pt")

sample = test_records[0]
f_texts, s_texts, _ = build_cross_encoder_pairs(sample["text"], sample["target_word"], catalog[sample["folder"]]["senses"])
enc = encode_pairs(f_texts, s_texts)
print(f"Gold sense: {sample['sense_num']}")
for i in range(min(3, len(f_texts))):
    print(f"Pair {i+1}: {f_texts[i]}  [SEP]  {s_texts[i]}")
print("\nTokens of pair 1:", tokenizer.convert_ids_to_tokens(enc["input_ids"][0])[:40])

lens = []
for r in random.sample(full_train, min(2000, len(full_train))):
    f, s, _ = build_cross_encoder_pairs(r["text"], r["target_word"], ALL_SENSES[r["folder"]])
    lens += [len(x) for x in tokenizer(f, s)["input_ids"]]
print(f"\nPair length: mean {sum(lens)/len(lens):.1f}, max {max(lens)} tokens (MAX_LEN={MAX_LEN})")

## Step 3: BanglaBERT cross-encoder (ELECTRA-base + 1-logit head)

In [ ]:
from transformers import AutoModelForSequenceClassification

def load_model(path=BASE_MODEL):
    return AutoModelForSequenceClassification.from_pretrained(path, num_labels=1).to(device)

model = load_model()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 4: Scoring & evaluation helpers + untrained baseline
Each sentence's candidates are scored, then put into a `[batch, max_senses]` matrix (padding = `-inf`) so softmax/cross-entropy runs over **that word's senses only**.

In [ ]:
import torch.nn.functional as F

def score_batch(model, items):
    firsts, seconds, rows, cols = [], [], [], []
    for row, (context, target, senses) in enumerate(items):
        f, s, nums = build_cross_encoder_pairs(context, target, senses)
        firsts += f; seconds += s
        rows += [row] * len(nums); cols += list(range(len(nums)))
    enc = encode_pairs(firsts, seconds).to(device)
    pair_scores = model(**enc).logits.squeeze(-1).float()
    matrix = torch.full((len(items), max(cols) + 1), float("-inf"), device=device)
    matrix[torch.tensor(rows, device=device), torch.tensor(cols, device=device)] = pair_scores
    return matrix

def to_items(batch):
    return [(r["text"], r["target_word"], ALL_SENSES[r["folder"]]) for r in batch]

@torch.no_grad()
def evaluate(model, records, bs=16):
    model.eval()
    loss_sum, preds = 0.0, []
    for i in range(0, len(records), bs):
        batch = records[i:i + bs]
        labels = torch.tensor([r["sense_label"] for r in batch], device=device)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = score_batch(model, to_items(batch))
        loss_sum += F.cross_entropy(logits, labels, reduction="sum").item()
        preds += logits.argmax(-1).tolist()
    gold = [r["sense_label"] for r in records]
    acc = sum(p == g for p, g in zip(preds, gold)) / len(records)
    return loss_sum / len(records), acc, preds

_, untrained_acc, _ = evaluate(model, val_records[:200])
print(f"Untrained accuracy on 200 val sentences: {untrained_acc*100:.2f}%")

## Step 5: Training (AdamW + linear warmup, FP16, early stopping on val accuracy)

In [ ]:
from transformers import get_linear_schedule_with_warmup

EPOCHS = 8
PATIENCE = 2          # stop after 2 epochs without val improvement
BATCH_SIZE = 8        # sentences per step (~20-30 (context, gloss) pairs)
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP = 0.10

no_decay = ("bias", "LayerNorm.weight")
param_groups = [
    {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(param_groups, lr=LR)
steps_per_epoch = (len(full_train) + BATCH_SIZE - 1) // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

checkpoint_dir = Path("../checkpoints/banglabert-indowordnet-v1")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print(f"Train sentences {len(full_train):,} | steps/epoch {steps_per_epoch} | max steps {total_steps}\n")
best_val_acc, bad_epochs, history = 0.0, 0, []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    order = list(range(len(full_train)))
    random.shuffle(order)
    run_loss, correct, seen = 0.0, 0, 0

    for step, start in enumerate(range(0, len(order), BATCH_SIZE), 1):
        batch = [full_train[i] for i in order[start:start + BATCH_SIZE]]
        labels = torch.tensor([r["sense_label"] for r in batch], device=device)

        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = score_batch(model, to_items(batch))
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        run_loss += loss.item() * len(batch)
        correct += (logits.argmax(-1) == labels).sum().item()
        seen += len(batch)
        if step % 500 == 0:
            print(f"  epoch {epoch} step {step}/{steps_per_epoch} | running loss {run_loss/seen:.4f} | acc {correct/seen*100:.1f}%")

    val_loss, val_acc, _ = evaluate(model, val_records)
    history.append({"epoch": epoch, "train_loss": run_loss / seen, "train_acc": correct / seen,
                    "val_loss": val_loss, "val_acc": val_acc})
    print(f">>> Epoch {epoch} ({time.time()-t0:.0f}s) | train loss {run_loss/seen:.4f} acc {correct/seen*100:.2f}% "
          f"| val loss {val_loss:.4f} acc {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc, bad_epochs = val_acc, 0
        model.save_pretrained(checkpoint_dir); tokenizer.save_pretrained(checkpoint_dir)
        print(f"    saved best checkpoint (val acc {val_acc*100:.2f}%)\n")
    else:
        bad_epochs += 1
        print(f"    no improvement ({bad_epochs}/{PATIENCE})\n")
        if bad_epochs >= PATIENCE:
            print("Early stopping.")
            break

## Step 6: Test-set evaluation

In [ ]:
best_model = load_model(checkpoint_dir)
test_loss, test_acc, test_preds = evaluate(best_model, test_records)
rand_b, first_b, _ = _baselines(test_records)

print(f"Test sentences:        {len(test_records):,}")
print(f"Random baseline:       {rand_b*100:.2f}%")
print(f"First-sense baseline:  {first_b*100:.2f}%")
print(f"Best val accuracy:     {best_val_acc*100:.2f}%")
print(f"TEST ACCURACY:         {test_acc*100:.2f}%  (loss {test_loss:.4f})\n")

by_k = collections.defaultdict(lambda: [0, 0])
for r, p in zip(test_records, test_preds):
    k = len(catalog[r["folder"]]["senses"])
    k = str(k) if k <= 5 else "6+"
    by_k[k][0] += p == r["sense_label"]; by_k[k][1] += 1
print("Accuracy by number of candidate senses:")
for k in sorted(by_k, key=lambda x: int(x.rstrip("+"))):
    c, n = by_k[k]
    print(f"  {k:>2} senses: {c/n*100:5.1f}%  ({n} sentences)")

## Step 7: Inference on new Bengali sentences
Raw sentences are marked automatically (inflected forms like `ফলের` are found), and candidate senses come from the catalog or directly from IndoWordNet (full glosses).

In [ ]:
def mark_target(sentence: str, target: str) -> str:
    if "**" in sentence:
        return sentence
    marked, found = mark_target_in_sentence(sentence, target, [])
    if not found:  # fall back: mark the first token containing the target
        marked = re.sub(rf"(\S*{re.escape(normalize_text_clean(target))}\S*)", r"**\1**", normalize_text_clean(sentence), count=1)
    return marked

def resolve_candidate_senses(target: str):
    t = normalize_text_clean(target)
    for v in catalog.values():
        if v["target_word"] == t:
            return v["senses"], "catalog"
    syns = filtered_synsets(iwn, t)
    if syns:
        return {str(i): full_gloss(s, t) for i, s in enumerate(syns, 1)}, "IndoWordNet"
    return None, "not found"

@torch.no_grad()
def predict_wsd(sentence: str, target: str, senses: dict = None, show=True):
    source = "given"
    if senses is None:
        senses, source = resolve_candidate_senses(target)
        if not senses:
            print(f"No senses found for '{target}'"); return None
    best_model.eval()
    t0 = time.time()
    marked = mark_target(sentence, target)
    f, s, nums = build_cross_encoder_pairs(marked, normalize_text_clean(target), senses)
    probs = torch.softmax(best_model(**encode_pairs(f, s).to(device)).logits.squeeze(-1).float(), -1).tolist()
    ranked = sorted(zip(nums, probs), key=lambda x: -x[1])
    if show:
        print(f"Target '{target}' ({source}) | {(time.time()-t0)*1000:.0f} ms")
        print(f"Input:  {prepare_context(marked)}")
        for rank, (n, p) in enumerate(ranked):
            print(f"{'>>>' if rank == 0 else '   '} Sense {n:2d} {p*100:5.1f}% | {senses[str(n)]}")
        print("-" * 70)
    return ranked

# 'ফল' is not in the 3k catalog (it was excluded as a baseline word) -> senses come from IndoWordNet
predict_wsd("সে ফলের দোকান থেকে এক কিলো পাকা আম কিনলো", "ফল")
predict_wsd("কঠোর পরিশ্রমের ফল সবসময় ভালোই হয়", "ফল")
for r in test_records[:3]:
    predict_wsd(r["text"], r["target_word"])
    print(f"    gold = sense {r['sense_num']}\n")

## Step 8: Interactive demo

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

target_input = widgets.Text(value="দড়ি", description="শব্দ:", layout=widgets.Layout(width="50%"))
sentence_input = widgets.Textarea(value="চাষি জমিতে মই দেওয়ার জন্য মই ও দড়ি নিয়ে এলো",
                                  description="বাক্য:", layout=widgets.Layout(width="90%", height="70px"))
check_btn = widgets.Button(description="Disambiguate", button_style="success", icon="search")
out_box = widgets.Output()

def on_check_clicked(_):
    with out_box:
        clear_output()
        if not sentence_input.value.strip() or not target_input.value.strip():
            print("অনুগ্রহ করে বাক্য এবং টার্গেট শব্দ উভয়ই প্রদান করুন।"); return
        predict_wsd(sentence_input.value.strip(), target_input.value.strip())

check_btn.on_click(on_check_clicked)
display(widgets.VBox([widgets.HTML("<h3>Bengali Word Sense Disambiguation</h3>"),
                      target_input, sentence_input, check_btn, out_box]))
on_check_clicked(None)